In [31]:
import pandas as pd
#Spark
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql import types
from pyspark.sql.functions import col
#General dependencies
import urllib.request
import os

In [32]:
pyspark.__file__

'/home/daniel/de_zoomcamp_2026_project/.venv/lib/python3.12/site-packages/pyspark/__init__.py'

In [ ]:
#configuration
gcs_credentials = '/home/daniel/de_zoomcamp_2026_project/gcs_credentials/credentials/service_account_creds.json'

conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('project_pipeline') \
    .set("spark.driver.memory", "4g") \
    .set("spark.executor.memory", "4g") \
    .set("spark.jars", "/home/daniel/de_zoomcamp_2026_project/gcs_hadoop_conn/gcs-connector-hadoop3-2.2.5.jar") \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", gcs_credentials)

In [34]:
#context
sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", gcs_credentials)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

In [37]:
#Creating Spark session
spark = SparkSession.builder \
        .config(conf= sc.getConf()) \
        .getOrCreate()

In [38]:
file_path = 'temp/weekly_housing_market_data_most_recent.tsv000.gz'
os.makedirs(
    os.path.dirname(file_path), 
    exist_ok= True
)

In [ ]:
# Getting historical data stored in aws s3
url = 'https://redfin-public-data.s3.us-west-2.amazonaws.com/redfin_covid19/weekly_housing_market_data_most_recent.tsv000.gz'

urllib.request.urlretrieve(
    url, 'temp/weekly_housing_market_data_most_recent.tsv000.gz'
)

In [ ]:
#Defining schema:
schema_housing = types.StructType([
    types.StructField('PERIOD_BEGIN', types.TimestampType(), True),
    types.StructField('PERIOD_END', types.TimestampType(), True),
    types.StructField('REGION_TYPE', types.StringType(), True),
    types.StructField('REGION_TYPE_ID', types.IntegerType(), True),
    types.StructField('REGION_NAME', types.StringType(), True),
    types.StructField('REGION_ID', types.IntegerType(), True),
    types.StructField('DURATION', types.StringType(), True),
    types.StructField('ADJUSTED_AVERAGE_NEW_LISTINGS', types.DoubleType(), True),
    types.StructField('ADJUSTED_AVERAGE_NEW_LISTINGS_YOY', types.DoubleType(), True),
    types.StructField('AVERAGE_PENDING_SALES_LISTING_UPDATES', types.DoubleType(), True),
    types.StructField('AVERAGE_PENDING_SALES_LISTING_UPDATES_YOY', types.DoubleType(), True),
    types.StructField('OFF_MARKET_IN_TWO_WEEKS', types.IntegerType(), True),
    types.StructField('OFF_MARKET_IN_TWO_WEEKS_YOY', types.DoubleType(), True),
    types.StructField('ADJUSTED_AVERAGE_HOMES_SOLD', types.DoubleType(), True),
    types.StructField('ADJUSTED_AVERAGE_HOMES_SOLD_YOY', types.DoubleType(), True),
    types.StructField('MEDIAN_NEW_LISTING_PRICE', types.DoubleType(), True),
    types.StructField('MEDIAN_NEW_LISTING_PRICE_YOY', types.DoubleType(), True),
    types.StructField('MEDIAN_SALE_PRICE', types.DoubleType(), True),
    types.StructField('MEDIAN_SALE_PRICE_YOY', types.DoubleType(), True),
    types.StructField('MEDIAN_DAYS_TO_CLOSE', types.DoubleType(), True),
    types.StructField('MEDIAN_DAYS_TO_CLOSE_YOY', types.DoubleType(), True),
    types.StructField('MEDIAN_NEW_LISTING_PPSF', types.DoubleType(), True),
    types.StructField('MEDIAN_NEW_LISTING_PPSF_YOY', types.DoubleType(), True),
    types.StructField('ACTIVE_LISTINGS', types.IntegerType(), True),
    types.StructField('ACTIVE_LISTINGS_YOY', types.DoubleType(), True),
    types.StructField('MEDIAN_DAYS_ON_MARKET', types.DoubleType(), True),
    types.StructField('MEDIAN_DAYS_ON_MARKET_YOY', types.DoubleType(), True),
    types.StructField('PERCENT_ACTIVE_LISTINGS_WITH_PRICE_DROPS', types.DoubleType(), True),
    types.StructField('PERCENT_ACTIVE_LISTINGS_WITH_PRICE_DROPS_YOY', types.DoubleType(), True),
    types.StructField('AGE_OF_INVENTORY', types.DoubleType(), True),
    types.StructField('AGE_OF_INVENTORY_YOY', types.DoubleType(), True),
    types.StructField('WEEKS_OF_SUPPLY', types.DoubleType(), True),
    types.StructField('WEEKS_OF_SUPPLY_YOY', types.DoubleType(), True),
    types.StructField('MEDIAN_PENDING_SQFT', types.DoubleType(), True),
    types.StructField('MEDIAN_PENDING_SQFT_YOY', types.DoubleType(), True),
    types.StructField('AVERAGE_SALE_TO_LIST_RATIO', types.DoubleType(), True),
    types.StructField('AVERAGE_SALE_TO_LIST_RATIO_YOY', types.DoubleType(), True),
    types.StructField('MEDIAN_SALE_PPSF', types.DoubleType(), True),
    types.StructField('MEDIAN_SALE_PPSF_YOY', types.DoubleType(), True),
    types.StructField('LAST_UPDATED', types.TimestampType(), True),
])

In [41]:
df_housing = \
    spark.read \
    .option('sep', '\t') \
    .option('header', 'true') \
    .schema(schema_housing) \
    .csv('temp/weekly_housing_market_data_most_recent.tsv000.gz')
             

In [23]:
df_housing.count()

5684424

In [42]:
print(df_housing.rdd.getNumPartitions)

<bound method RDD.getNumPartitions of MapPartitionsRDD[4] at javaToPython at NativeMethodAccessorImpl.java:0>


In [43]:
df_housing \
    .repartition(10) \
    .write \
    .mode('overwrite') \
    .parquet('gs://de-zoomcamp-2026-project-bucket/weekly_housing_market_data_most_recent.parquet')

In [44]:
spark.stop()